# H286-Gruppenversuch: kleiner Starter

**A100 → Alle ausführen.** In der Dateiauswahl das bereitgestellte **h286_group_restore_paket.zip** auswählen.
Danach laufen alle Gruppen- und Antwortstufen automatisch. Kein Drive und kein Warten auf eine GPU.
Am Ende Ergebnis-/Diagnose-ZIP herunterladen. Das private Paket nicht öffentlich hochladen.
In normalem Jupyter die Paket-ZIP neben diesen Starter legen.


In [ ]:
import hashlib, io, json, traceback, zipfile
from pathlib import Path
PACKAGE_SHA = '647a010038177faeda0106785db7b44ef44ebacd1d4cfe306cbbb794e6adc267'
NOTEBOOK_SHA = 'd26a40913f39b6693da2fed968d41f4c1752155c9fd6de03eb075d0eec0b02a5'
NOTEBOOK_BYTES = 910154
MEMBER = 'h286_group_restore.ipynb'
PAKET = Path('h286_group_restore_paket.zip')
stage, started = 'Paket laden', 0

def anbieten(path):
    try:
        from google.colab import files
        files.download(str(path))
    except Exception:
        try:
            from IPython.display import FileLink, display
            display(FileLink(str(path)))
        except Exception:
            print('Datei:', path)

try:
    if PAKET.is_file():
        if PAKET.stat().st_size > 2*1024**2:
            raise ValueError('Paket zu groß.')
        raw = PAKET.read_bytes()
    else:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError('h286_group_restore_paket.zip neben den Starter legen.') from None
        print('Jetzt genau das heruntergeladene h286_group_restore_paket.zip auswählen.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Bitte genau eine Paket-ZIP auswählen.')
        raw = bytes(next(iter(uploaded.values())))
    stage = 'ZIP prüfen'
    if len(raw) > 2*1024**2 or hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
        raise ValueError('Paket-Prüfsumme stimmt nicht; Originalpaket verwenden.')
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        members = archive.infolist()
        if (len(members) != 1 or members[0].filename != MEMBER or members[0].is_dir()
            or members[0].flag_bits & 1 or members[0].file_size != NOTEBOOK_BYTES
            or members[0].file_size > 2*1024**2):
            raise ValueError('Unerwarteter oder zu großer ZIP-Inhalt.')
        if archive.testzip() is not None:
            raise ValueError('ZIP-CRC-Prüfung fehlgeschlagen.')
        notebook = archive.read(members[0])
    stage = 'Notebook prüfen'
    if hashlib.sha256(notebook).hexdigest() != NOTEBOOK_SHA:
        raise ValueError('Notebook-Prüfsumme stimmt nicht.')
    document = json.loads(notebook)
    if document.get('nbformat') != 4 or not isinstance(document.get('cells'), list):
        raise ValueError('Ungültiges Notebookformat.')
    cells = [cell for cell in document['cells'] if cell.get('cell_type') == 'code']
    if len(cells) != 3:
        raise ValueError('Unerwartete Anzahl Original-Codezellen.')
    scope = {'__name__': '__main__'}
    for number, cell in enumerate(cells, 1):
        stage = 'Original-Codezelle ' + str(number)
        source = cell['source']
        if isinstance(source, list):
            source = ''.join(source)
        if not isinstance(source, str):
            raise ValueError('Ungültige Codezelle.')
        print('Original-Codezelle', number, '/', len(cells), 'startet.', flush=True)
        started += 1
        exec(compile(source, 'h286_group_restore_cell_' + str(number), 'exec'), scope)
except BaseException as error:
    if (started == 3 and isinstance(error, RuntimeError)
        and str(error).startswith('Versuch angehalten;') and 'scope' in globals()
        and scope.get('failure') is not None and isinstance(scope.get('target'), Path)
        and scope['target'].is_file()):
        print('Originalversuch angehalten. Sein vollständiges Diagnose-ZIP wurde bereits angeboten:', scope['target'])
        raise RuntimeError('Bitte das bereits angebotene Original-Diagnose-ZIP verwenden.') from None
    status = {'state': 'failed', 'stage': stage, 'code_cells_started': started,
              'error_type': type(error).__name__, 'error': str(error)[:2000],
              'expected_package_sha256': PACKAGE_SHA, 'expected_notebook_sha256': NOTEBOOK_SHA}
    target = Path('h286_group_restore_START_diagnose.zip')
    with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_DEFLATED) as diagnostic:
        diagnostic.writestr('bootstrap_status.json', json.dumps(status, ensure_ascii=False, indent=2))
        diagnostic.writestr('traceback.txt', traceback.format_exc()[-12000:])
    print('Starter angehalten bei:', stage, '| Diagnose:', target)
    anbieten(target)
    raise RuntimeError('Starter angehalten; das Diagnose-ZIP enthält den Fehler.') from None
